# Final Project Data Mining Minggu 1
Uji baseline YOLO26n dan pembanding YOLO26s sesuai starter dosen. Kaggle CPU menjalankan pelatihan dari bobot awal saat GPU mengantre. Kedua model memakai data UTS olahan 640, anggaran lima epoch, batch 8, seed 42 dan evaluasi val yang sama. E1 di notebook ini menguji arsitektur. E1 historis menguji pembekuan 10 lapisan YOLO26n.

Ekspor 1280 dibuat dari gambar olahan 640 untuk verifikasi alur. Eksperimen resolusi dari citra asli berjalan terpisah di E2. Input di versi ini hanya memuat train dan validation.

## 1. Pasang dependensi dan siapkan input train dan validation

In [ ]:
%pip install -q ultralytics==8.4.210 pycocotools==2.0.11
from collections import Counter
from pathlib import Path, PurePosixPath
import csv
import json
import math
import os
import shutil
import tempfile
import zipfile

from PIL import Image, ImageOps
import yaml

anchors = list(Path("/kaggle/input").rglob("coco_val_gt.json"))
assert len(anchors) == 1, f"Harus tersedia satu input train/val olahan: {anchors}"
asset = anchors[0].parent
archive = asset / "processed_train_val.zip"
INPUT_PATH = str(archive if archive.is_file() else asset)
OUTPUT_DIR = Path("/kaggle/working/fp_dm_m1")
WORK_DATA_DIR = Path(tempfile.gettempdir()) / (OUTPUT_DIR.name + "_data")
# Isi jika ada hasil ekspor 1280 dari citra asli dengan nama file dan split yang sama.
NATIVE_1280_PATH = ""
SOURCE_ALREADY_RESIZED = True  # Ubah ke False jika INPUT_PATH berisi citra asli.
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SPLIT_ALIASES = {"train": ("train",), "val": ("val", "valid"), "test": ("test",)}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DATA_DIR.mkdir(parents=True, exist_ok=True)


def extract_zip(archive, cache_name):
    cache = WORK_DATA_DIR / "input_cache" / cache_name
    signature = [str(archive.resolve()), archive.stat().st_size, archive.stat().st_mtime_ns]
    marker = cache / ".archive_signature.json"
    if marker.is_file() and json.loads(marker.read_text()) == signature:
        return cache
    cache.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=cache.parent) as temp:
        staging = Path(temp) / "dataset"
        staging.mkdir()
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                path = PurePosixPath(member.filename)
                if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
                    raise ValueError(f"Nama file ZIP tidak valid: {member.filename}")
            zipped.extractall(staging)
        (staging / marker.name).write_text(json.dumps(signature))
        if cache.exists():
            shutil.rmtree(cache)
        staging.rename(cache)
    return cache


def split_folders(root, split):
    for alias in SPLIT_ALIASES[split]:
        layouts = ((root / "images" / alias, root / "labels" / alias),
                   (root / alias / "images", root / alias / "labels"))
        for images, labels in layouts:
            if images.is_dir() and labels.is_dir():
                return images, labels
    return None


def locate_dataset(value, cache_name):
    path = Path(value).expanduser() if value else Path("/kaggle/input")
    if path.is_file():
        if path.suffix.lower() != ".zip":
            raise ValueError(f"Gunakan folder dataset atau ZIP: {path}")
        path = extract_zip(path, cache_name)
    if not path.is_dir():
        raise FileNotFoundError(f"Input tidak ditemukan: {path}")

    possible = {path}
    for candidate in path.rglob("train"):
        if candidate.is_dir():
            possible.update((candidate.parent, candidate.parent.parent))
    matches = sorted(root for root in possible if split_folders(root, "train") and split_folders(root, "val"))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ada beberapa dataset. Isi INPUT_PATH atau NATIVE_1280_PATH dengan satu folder: {matches}")
    archives = sorted(path.rglob("*.zip"))
    if len(archives) == 1 and not (path / ".archive_signature.json").exists():
        return locate_dataset(str(archives[0]), cache_name)
    raise FileNotFoundError(f"Folder YOLO train/val belum ditemukan. Isi jalur ZIP atau folder. ZIP yang terlihat: {archives[:8]}")


BASE_ROOT = locate_dataset(INPUT_PATH, "base")
NATIVE_ROOT = locate_dataset(NATIVE_1280_PATH, "native1280") if NATIVE_1280_PATH else None
print("Dataset baseline:", BASE_ROOT)
print("Ekspor 1280 dari citra asli:", NATIVE_ROOT or "belum ditambahkan")

## 2. Validasi label dan hitung profil ukuran
`profil_ukuran_box.csv` berisi ukuran setiap kotak. Kolom `sisi_min_640_px` mengukur sisi terpendek bila gambar di-letterbox ke 640. Ukuran yang dipakai untuk kategori COCO dihitung pada koordinat citra sumber validation.

In [ ]:
def names_from(root):
    for filename in ("data.yaml", "dataset.yaml", "data.yml"):
        if (root / filename).is_file():
            data = yaml.safe_load((root / filename).read_text(encoding="utf-8"))
            names = data.get("names", [])
            if isinstance(names, list) and names:
                return dict(enumerate(names))
            if isinstance(names, dict) and names:
                return {int(key): str(value) for key, value in names.items()}
    print(f"File nama kelas tidak ditemukan di {root}. Memakai kelas UTS 0=infected.")
    return {0: "infected"}


def read_boxes(path, names):
    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"{path}:{line_number} harus berisi class x y w h.")
        category, x, y, w, h = map(float, parts)
        if not math.isfinite(category) or category not in names:
            raise ValueError(f"{path}:{line_number} kelas tidak dikenal.")
        if not all(map(math.isfinite, (x, y, w, h))) or w <= 0 or h <= 0:
            raise ValueError(f"{path}:{line_number} ukuran kotak tidak valid.")
        x0, y0, x1, y1 = x - w/2, y - h/2, x + w/2, y + h/2
        if min(x0, y0) < -0.00001 or max(x1, y1) > 1.00001:
            raise ValueError(f"{path}:{line_number} kotak keluar dari gambar.")
        yield int(category), max(0.0, x0), max(0.0, y0), min(1.0, x1), min(1.0, y1)


def inventory(root, names):
    records = []
    for split in SPLIT_ALIASES:
        pair = split_folders(root, split)
        if not pair:
            continue
        images, labels = pair
        keys = set()
        for image in sorted((p for p in images.rglob("*") if p.suffix.lower() in IMAGE_EXTS), key=lambda path: path.as_posix().casefold()):
            key = image.relative_to(images).with_suffix("")
            if key in keys:
                raise ValueError(f"Nama gambar ganda dalam split {split}: {key}")
            keys.add(key)
            label = labels / key.with_name(key.name + ".txt")
            if not label.is_file():
                raise FileNotFoundError(f"Label hilang: {label}. Label kosong tetap sah untuk background.")
            with Image.open(image) as source:
                width, height = ImageOps.exif_transpose(source).size
            records.append((split, key, image, width, height, list(read_boxes(label, names))))
        if split in ("train", "val") and not keys:
            raise ValueError(f"Split {split} kosong.")
    return records


def write_csv(path, columns, rows):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


NAMES = names_from(BASE_ROOT)
BASE = inventory(BASE_ROOT, NAMES)
assert Counter(row[0] for row in BASE) == {"train": 1244, "val": 356}
assert len({row[1] for row in BASE}) == 1600
assert sum(len(row[-1]) for row in BASE if row[0] == "train") == 10540
assert sum(len(row[-1]) for row in BASE if row[0] == "val") == 2205
rows = []
for split, key, image, width, height, boxes in BASE:
    scale = 640 / max(width, height)
    for category, x0, y0, x1, y1 in boxes:
        w, h = (x1 - x0) * width, (y1 - y0) * height
        rows.append({"split": split, "gambar": key.as_posix(), "kelas": NAMES[category],
                     "lebar_px": w, "tinggi_px": h, "luas_px2": w*h,
                     "sisi_min_640_px": min(w, h)*scale})
write_csv(OUTPUT_DIR / "profil_ukuran_box.csv",
          ["split", "gambar", "kelas", "lebar_px", "tinggi_px", "luas_px2", "sisi_min_640_px"], rows)

summary = []
for split in SPLIT_ALIASES:
    items = [item for item in BASE if item[0] == split]
    boxes = [row for row in rows if row["split"] == split]
    if items:
        small = sum(row["sisi_min_640_px"] < 8 for row in boxes)
        summary.append({"split": split, "gambar": len(items), "box": len(boxes),
                        "box_sisi_min_640_di_bawah_8": small,
                        "persen_box_di_bawah_8": round(100 * small / len(boxes), 2) if boxes else ""})
write_csv(OUTPUT_DIR / "ringkasan_profil_box.csv",
          ["split", "gambar", "box", "box_sisi_min_640_di_bawah_8", "persen_box_di_bawah_8"], summary)
for record in summary:
    print(record)

## 3. Ekspor 1280 dan periksa split
Anotasi ikut diskalakan dan digeser sesuai letterbox. Bila tersedia, `NATIVE_1280_PATH` menggunakan hasil ekspor dari citra asli dengan split, nama file, dan jumlah kotak yang cocok dengan baseline. Jalur data baseline dipakai sebagai acuan validasi bersama untuk E0 dan E1.

In [ ]:
def write_yolo_label(path, boxes):
    with path.open("w", encoding="utf-8") as handle:
        for category, x0, y0, x1, y1 in boxes:
            handle.write(f"{category} {(x0+x1)/2:.8f} {(y0+y1)/2:.8f} {x1-x0:.8f} {y1-y0:.8f}\n")


def build_dataset(records, name, size=None):
    destination = WORK_DATA_DIR / name
    with tempfile.TemporaryDirectory(dir=WORK_DATA_DIR) as temp:
        staging = Path(temp) / name
        for split, key, image, width, height, boxes in records:
            image_folder = staging / "images" / split / key.parent
            label_folder = staging / "labels" / split / key.parent
            image_folder.mkdir(parents=True, exist_ok=True)
            label_folder.mkdir(parents=True, exist_ok=True)
            if size is None:
                image_output = image_folder / (key.name + image.suffix.lower())
                try:
                    os.link(image, image_output)
                except OSError:
                    shutil.copyfile(image, image_output)
                updated = boxes
            else:
                scale = size / max(width, height)
                nw, nh = max(1, round(width * scale)), max(1, round(height * scale))
                px, py = (size - nw) // 2, (size - nh) // 2
                with Image.open(image) as source:
                    rgb = ImageOps.exif_transpose(source).convert("RGB")
                    padded = Image.new("RGB", (size, size), (114, 114, 114))
                    padded.paste(rgb.resize((nw, nh), Image.Resampling.LANCZOS), (px, py))
                    padded.save(image_folder / (key.name + ".jpg"), quality=95)
                updated = [(c, (x0*nw+px)/size, (y0*nh+py)/size, (x1*nw+px)/size, (y1*nh+py)/size)
                           for c, x0, y0, x1, y1 in boxes]
            write_yolo_label(label_folder / (key.name + ".txt"), updated)
        data = {"path": str(destination), "names": NAMES}
        data.update({split: f"images/{split}" for split, *_ in records})
        (staging / "data.yaml").write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")
        if destination.exists():
            shutil.rmtree(destination)
        staging.rename(destination)
    return destination / "data.yaml"


EVAL_YAML = build_dataset(BASE, "dataset_baseline")
if NATIVE_ROOT:
    if names_from(NATIVE_ROOT) != NAMES:
        raise ValueError("Nama kelas pada baseline dan dataset 1280 berbeda.")
    native = inventory(NATIVE_ROOT, NAMES)
    signature = lambda items: {(s, key): Counter(b[0] for b in boxes) for s, key, _, _, _, boxes in items}
    if signature(native) != signature(BASE):
        raise ValueError("Split, nama file, atau jumlah kotak per kelas berbeda pada dua dataset.")
    if any((width, height) != (1280, 1280) for _, _, _, width, height, _ in native):
        raise ValueError("NATIVE_1280_PATH harus berisi ekspor 1280 × 1280.")
    EXPORT_YAML = build_dataset(native, "dataset_1280")
    source_note = "Ekspor 1280 yang dipasok kelompok. Split dan jumlah kotak diperiksa."
else:
    EXPORT_YAML = build_dataset(BASE, "dataset_1280", size=1280)
    source_note = ("Ekspor 1280 dari dataset yang sebelumnya sudah diubah ukurannya."
                   if SOURCE_ALREADY_RESIZED else "Ekspor 1280 dari input yang dinyatakan sebagai citra asli.")
def archive_export(data_yaml):
    root = data_yaml.parent
    target = OUTPUT_DIR / "dataset_1280.zip"
    staging = target.with_suffix(".tmp")
    portable = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    portable.pop("path")
    with zipfile.ZipFile(staging, "w", compression=zipfile.ZIP_STORED) as archive:
        archive.writestr("dataset_1280/data.yaml", yaml.safe_dump(portable, sort_keys=False))
        for item in sorted(root.rglob("*")):
            if item.is_file() and item != data_yaml:
                archive.write(item, item.relative_to(root.parent))
    staging.replace(target)
    return target


EXPORT_ZIP = archive_export(EXPORT_YAML)
export_summary = {"asal": source_note, "total_gambar": len(BASE),
                  "split": dict(Counter(record[0] for record in BASE)),
                  "total_box": sum(len(record[-1]) for record in BASE),
                  "resolusi": [1280, 1280], "arsip": str(EXPORT_ZIP)}
(OUTPUT_DIR / "ringkasan_ekspor_1280.json").write_text(json.dumps(export_summary, indent=2), encoding="utf-8")
print(json.dumps(export_summary, indent=2))

## 4. Siapkan anotasi COCO untuk split val
`coco_val_gt.json` menyimpan ground truth. Evaluasi memakai kategori `small` dari COCOeval dengan koordinat citra validation acuan yang sama untuk seluruh run. Angka AP-small baru dihasilkan setelah ada prediksi model.

In [ ]:
def write_coco_validation(records):
    data = {"info": {"description": "FP Data Mining validation"}, "licenses": [],
            "images": [], "annotations": [],
            "categories": [{"id": key+1, "name": name} for key, name in sorted(NAMES.items())]}
    image_paths = []
    for split, key, image, width, height, boxes in records:
        if split != "val":
            continue
        image_id = len(image_paths) + 1
        image_paths.append(image)
        data["images"].append({"id": image_id, "file_name": key.as_posix() + image.suffix.lower(),
                               "width": width, "height": height})
        for category, x0, y0, x1, y1 in boxes:
            w, h = (x1-x0)*width, (y1-y0)*height
            data["annotations"].append({"id": len(data["annotations"])+1, "image_id": image_id,
                                        "category_id": category+1, "bbox": [x0*width, y0*height, w, h],
                                        "area": w*h, "iscrowd": 0})
    if not data["annotations"]:
        raise ValueError("Split val tidak memiliki anotasi objek.")
    path = OUTPUT_DIR / "coco_val_gt.json"
    path.write_text(json.dumps(data), encoding="utf-8")
    return path, image_paths


COCO_GT, VAL_IMAGES = write_coco_validation(BASE)
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

acuan = COCO(str(COCO_GT))
paket = json.loads((asset / "coco_val_gt.json").read_text())
assert all(acuan.dataset[key] == paket[key] for key in ("images", "annotations", "categories"))
sempurna = [{k: row[k] for k in ("image_id", "category_id", "bbox")} | {"score": 1.0}
            for row in acuan.dataset["annotations"]]
uji = COCOeval(acuan, acuan.loadRes(sempurna), "bbox")
uji.evaluate()
uji.accumulate()
uji.summarize()
assert math.isclose(float(uji.stats[3]), 1.0)
(OUTPUT_DIR / "uji_cocoeval.json").write_text(json.dumps({
    "images": len(acuan.imgs), "boxes": len(acuan.anns), "perfect_prediction_ap_small": float(uji.stats[3])
}))

## 5. Jalankan training dan catat hasil
Jika paket belum tersedia, jalankan `%pip install -q ultralytics pycocotools` pada sel baru di Kaggle. Untuk menjalankan baseline, isi `RUN_IDS = ("E0",)` dan `EPOCHS` sesuai arahan dosen. Konfigurasi E1 pada template ini masih kosong; untuk mengulang acuan M1 `freeze=10`, lihat konfigurasi pada [notebook lokal](FP_Data_Mining_M1_Eksekusi_Lokal.ipynb). Jika E0 sudah selesai, pilih hanya E1 pada eksekusi berikutnya. Isi `CHECKPOINTS` bila bobot kelompok sudah ada dan tinggal dievaluasi. Waktu training bergantung pada GPU, resolusi, dan jumlah epoch. Run uji coba satu epoch berguna untuk cek alur dan sebaiknya memakai folder output terpisah dari run yang akan dilaporkan.

In [ ]:
import hashlib
import platform
import torch
import ultralytics
from importlib.metadata import version

cpus = sorted(os.sched_getaffinity(0))[:4]
assert len(cpus) == 4
os.sched_setaffinity(0, cpus)
torch.set_num_threads(4)
print("Training starts on Kaggle CPU, logical cores:", cpus, flush=True)
RUN_IDS = ("E0", "E1")
EPOCHS = 5
BATCH = 8
SEED = 42
DEVICE = "cpu"
WEIGHTS = str(asset / "yolo26n.pt")
E1 = {"perlakuan": "Pembanding YOLO26s, imgsz 640", "imgsz": 640,
      "data": EVAL_YAML, "weights": "yolo26s.pt", "train_args": {}}
CHECKPOINTS = {}

def ap_small_for_model(model, size, device, coco_gt, val_images, run_dir):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    lookup = {path.resolve(): i for i, path in enumerate(val_images, 1)}
    seen = set()
    preds = []
    manifest = run_dir / "coco_val_inputs.txt"
    manifest.write_text("\n".join(str(path.resolve()) for path in val_images) + "\n", encoding="utf-8")
    for result in model.predict(source=str(manifest), imgsz=size, device=device,
                                batch=1, conf=0.001, max_det=100, stream=True, verbose=False):
        image_id = lookup[Path(result.path).resolve()]
        if image_id in seen:
            raise ValueError(f"Prediksi berulang untuk ID gambar {image_id}.")
        seen.add(image_id)
        for (x0, y0, x1, y1), score, category in zip(
                result.boxes.xyxy.cpu().tolist(), result.boxes.conf.cpu().tolist(),
                result.boxes.cls.cpu().tolist(), strict=True):
            preds.append({"image_id": image_id, "category_id": int(category)+1,
                          "bbox": [x0, y0, x1-x0, y1-y0], "score": score})
    if len(seen) != len(lookup):
        raise ValueError("Prediksi validation tidak lengkap.")
    (run_dir / "coco_val_predictions.json").write_text(json.dumps(preds), encoding="utf-8")
    ground_truth = COCO(str(coco_gt))
    if preds:
        predictions = ground_truth.loadRes(preds)
    else:
        predictions = COCO()
        predictions.dataset = {"images": ground_truth.dataset["images"],
                               "categories": ground_truth.dataset["categories"], "annotations": []}
        predictions.createIndex()
    scorer = COCOeval(ground_truth, predictions, "bbox")
    scorer.evaluate()
    scorer.accumulate()
    scorer.summarize()
    (run_dir / "cocoeval_val.json").write_text(json.dumps({"AP_small": float(scorer.stats[3])}), encoding="utf-8")
    assert scorer.stats[3] >= 0, "Tidak ada anotasi COCO small yang dapat dinilai."
    return float(scorer.stats[3])


def log_completed(path, row):
    previous = []
    if path.is_file():
        with path.open(encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            if reader.fieldnames != list(row):
                raise ValueError("Format eksperimen.csv berbeda. Pakai folder output terpisah.")
            previous = [entry for entry in reader if entry["id"] != row["id"]]
    write_csv(path, list(row), sorted([*previous, row], key=lambda item: item["id"]))


def train_selected():
    if not RUN_IDS:
        print("Persiapan data selesai. Isi RUN_IDS dan EPOCHS untuk menjalankan training.")
        return
    configs = {"E0": {"perlakuan": "Baseline YOLO26n, imgsz 640", "imgsz": 640,
                      "data": EVAL_YAML, "train_args": {}}, "E1": E1}
    if any(run not in CHECKPOINTS for run in RUN_IDS) and (not isinstance(EPOCHS, int) or EPOCHS < 1):
        raise ValueError("Isi EPOCHS dengan jumlah epoch sesuai rancangan eksperimen.")
    for run in RUN_IDS:
        if not configs.get(run):
            raise ValueError(f"Konfigurasi {run} belum diisi.")
    from ultralytics import YOLO
    for run in RUN_IDS:
        config = configs[run]
        size = int(config["imgsz"])
        project = OUTPUT_DIR / "runs"
        existing = project / run
        if existing.exists():
            raise FileExistsError(f"{existing} sudah ada. Simpan folder run itu sebelum mengulang konfigurasi.")
        train_options = {"data": str(config["data"]), "epochs": EPOCHS, "imgsz": size, "device": DEVICE,
                         "batch": BATCH, "seed": SEED, "project": str(project), "name": run}
        options = config.get("train_args", {})
        if train_options.keys() & options.keys():
            raise ValueError("Ubah konfigurasi utama E1 secara langsung, bukan melalui train_args.")
        if run in CHECKPOINTS:
            checkpoint = Path(CHECKPOINTS[run])
            if not checkpoint.is_file():
                raise FileNotFoundError(f"Bobot {run} tidak ditemukan: {checkpoint}")
            run_dir = existing
            run_dir.mkdir(parents=True)
        else:
            model = YOLO(config.get("weights", WEIGHTS))
            model.train(**train_options, **options, workers=0, deterministic=True)
            run_dir = Path(model.trainer.save_dir)
            checkpoint = run_dir / "weights" / "best.pt"
        trained = YOLO(str(checkpoint))
        metrics = trained.val(data=str(EVAL_YAML), split="val", imgsz=size, device=DEVICE,
                              plots=False, max_det=100, project=str(run_dir), name="validation")
        small_ap = ap_small_for_model(trained, size, DEVICE, COCO_GT, VAL_IMAGES, run_dir)
        log_completed(OUTPUT_DIR / "eksperimen.csv", {
            "id": run, "perlakuan": config["perlakuan"], "split": "val",
            "presisi": float(metrics.box.mp), "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map),
            "AP-small": small_ap, "latensi_cpu_ms": "",
        })
        print(f"Hasil validation {run} tersimpan di {OUTPUT_DIR / 'eksperimen.csv'}.")

train_selected()

## 6. Bukti untuk laporan
- **Profil ukuran box** · `ringkasan_profil_box.csv` dan `profil_ukuran_box.csv`.
- **Ekspor ulang 1280** · `ringkasan_ekspor_1280.json` dan `dataset_1280.zip`. Catat sumber citra.
- **COCOeval** · `coco_val_gt.json` membuktikan persiapan anotasi val. `runs/E0/cocoeval_val.json` atau keluaran E1 mencatat AP-small sesudah evaluasi.
- **Training** · `runs/E0`, `runs/E1`, dan `eksperimen.csv` setelah run selesai. Jika AP-small kosong, kategori small tidak dapat dievaluasi pada set tersebut. Latensi CPU diisi setelah pengukuran.
- Arsip `dataset_1280.zip` bisa diunduh dari output Kaggle dan dipasang sebagai input privat untuk notebook berikutnya.

In [ ]:
def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

with (OUTPUT_DIR / "eksperimen.csv").open() as handle:
    metrics = list(csv.DictReader(handle))
assert [row["id"] for row in metrics] == list(RUN_IDS)
for run in RUN_IDS:
    with (OUTPUT_DIR / f"runs/{run}/results.csv").open() as handle:
        assert sum(1 for _ in csv.DictReader(handle)) == EPOCHS
assert len({row["id"] for row in metrics}) == 2

import statistics
from itertools import islice
import numpy as np
from ultralytics import YOLO

available = sorted(os.sched_getaffinity(0))
cpus = available[:min(4, len(available))]
os.sched_setaffinity(0, cpus)
latency = {}
for row in metrics:
    run_dir = OUTPUT_DIR / "runs" / row["id"]
    model = YOLO(str(run_dir / "weights/best.pt"))
    options = dict(source=str(run_dir / "coco_val_inputs.txt"), imgsz=640, device="cpu", batch=1,
                   conf=0.05, max_det=100, stream=True, verbose=False)
    torch.set_num_threads(len(cpus))
    warmup = model.predict(**options)
    list(islice(warmup, 10))
    warmup.close()
    torch.set_num_threads(len(cpus))
    times = [sum(result.speed.values()) for result in model.predict(**options)]
    assert len(times) == 356
    latency[row["id"]] = {"n": len(times), "warmup": 10, "threads": len(cpus),
                          "median_ms": statistics.median(times), "p95_ms": float(np.percentile(times, 95))}
    row["latensi_cpu_ms"] = latency[row["id"]]["median_ms"]
    log_completed(OUTPUT_DIR / "eksperimen.csv", row)
    write_csv(run_dir / "latensi_cpu.csv", ["id_urutan", "latensi_ms"],
              [{"id_urutan": i, "latensi_ms": value} for i, value in enumerate(times, 1)])
    print(row)
(OUTPUT_DIR / "latensi_cpu.json").write_text(json.dumps(latency, indent=2))

evidence = {
    "stage": "M1", "training_performed": True, "epochs": EPOCHS, "batch": BATCH, "seed": SEED,
    "device": f"Kaggle CPU, {len(cpus)} threads", "python": platform.python_version(),
    "torch": torch.__version__, "ultralytics": ultralytics.__version__, "pycocotools": version("pycocotools"),
    "input_ground_truth_sha256": digest(asset / "coco_val_gt.json"), "coco_sha256": digest(COCO_GT),
    "checkpoints_sha256": {run: digest(OUTPUT_DIR / f"runs/{run}/weights/best.pt") for run in RUN_IDS}
}
(OUTPUT_DIR / "run.json").write_text(json.dumps(evidence, indent=2))
print(json.dumps(evidence, indent=2))